In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
listing_path = Path("../data/mortgage/CRMLSListing_202401_202605_with_mortgage.csv")
sold_path = Path("../data/mortgage/CRMLSSold_202401_202605_with_mortgage.csv")

In [ ]:
listing = pd.read_csv(listing_path)
listing

In [ ]:
sold = pd.read_csv(sold_path)
sold

In [ ]:
listing.columns

In [ ]:
sold.columns

### Note on metadata fields:
- Keys: ListingKey, ListingKeyNumeric, ListingId
- Address: UnparsedAddress, StreetNumberNumeric
- Agent/list: ListAgentEmail, ListAgentFirstName, ListAgentLastName, ListAgentFullName, ListOfficeName
- Buyer-side: BuyerAgentFirstName, BuyerAgentLastName, BuyerAgentMlsId, BuyerOfficeName, BuyerOfficeAOR, BuyerAgentAOR
- Location (State): StateOrProvince

### Note on market analysis fields:
- Price: OriginalListPrice, ListPrice, ClosePrice
- Timing: CloseDate, PurchaseContractDate, ListingContractDate, DaysOnMarket, year_month
- Geography: CountyOrParish, City, PostalCode, MLSAreaMajor, HighSchoolDistrict
- Location (coordinates): Latitude, Longitude
- Property Details: PropertySubType, LivingArea, BedroomsTotal, BathroomsTotalInteger, YearBuilt, LotSizeSquareFeet, LotSizeAcres, GarageSpaces, ParkingTotal, AttachedGarageYN, Stories, Levels, MainLevelBedrooms, AssociationFee, NewConstructionYN, FireplaceYN, iewYN, PoolPrivateYN, Flooring

### Removing duplicate columns

In [ ]:
def clean_duplicate_columns(df):
    columns = df.columns
    for column in columns:
        if column.endswith(".1"):
            df.drop(column, axis=1, inplace=True)
    return df

listing = clean_duplicate_columns(listing)

### Converting date columns into datetime

In [ ]:
def to_datetime(df, date_columns): 

    df[date_columns] = df[date_columns].apply(pd.to_datetime, errors="coerce")

    # df["year_month"] = pd.to_datetime(
    #     df["year_month"], format="%Y-%m", errors="coerce")
    # df["Month"] = df["year_month"].dt.month
    # df["Year"] = df["year_month"].dt.year

    return df

In [ ]:
listing_date_columns = ["ListingContractDate", "ContractStatusChangeDate"]
sold_date_columns = date_columns = [
        "ListingContractDate",
        "PurchaseContractDate",
        "CloseDate",
        "ContractStatusChangeDate",
    ]

In [ ]:
listing = to_datetime(listing, listing_date_columns)
sold = to_datetime(sold, sold_date_columns)

### Flagging date consistency violations

In [ ]:
sold["listing_after_close_flag"] = sold["ListingContractDate"] > sold["CloseDate"]
sold["purchase_after_close_flag"] = sold["PurchaseContractDate"] > sold["CloseDate"]
sold["negative_timeline_flag"] = sold["ListingContractDate"] > sold["PurchaseContractDate"]

timeline_flag_columns = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag",
]
sold[timeline_flag_columns].sum()

### Dropping redundant list agent columns

In [ ]:
def list_agent(df):
    columns = ["ListAgentEmail", "ListAgentFirstName", "ListAgentLastName"]
    df.drop(columns=columns, inplace=True)
    return df

In [ ]:
listing = list_agent(listing)
sold = list_agent(sold)

### Filtering coordinates to purely California

In [ ]:
city_path = Path("../data/city_boundaries/City_and_County_Boundaries.geojson")
city_gdf = gpd.read_file(city_path)
city_gdf = city_gdf.set_crs("EPSG:4326")

california_boundary = city_gdf.dissolve()

listing_gdf = gpd.GeoDataFrame(
    listing, 
    geometry=gpd.points_from_xy(listing.Longitude, listing.Latitude))

fig, ax = plt.subplots(figsize=(10, 12))

california_boundary.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1,
)

listing_gdf.plot(
    ax=ax,
    markersize=4,
    alpha=0.4,
    legend=True,
)

ax.set_title("Listing Coordinates Over California Boundary")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

In [ ]:
sold_gdf = gpd.GeoDataFrame(
    sold, 
    geometry=gpd.points_from_xy(sold.Longitude, sold.Latitude))

fig, ax = plt.subplots(figsize=(10, 12))

california_boundary.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1,
)

sold_gdf.plot(
    ax=ax,
    markersize=4,
    alpha=0.4,
    legend=True,
)

ax.set_title("Sold Coordinates Over California Boundary")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

In [ ]:
listing["Longitude"] = listing["Longitude"].apply(lambda x: -x if x > 0 else x) 
sold["Longitude"] = sold["Longitude"].apply(lambda x: -x if x > 0 else x)

In [ ]:
listing_gdf = gpd.GeoDataFrame(
    listing, 
    geometry=gpd.points_from_xy(listing.Longitude, listing.Latitude))

fig, ax = plt.subplots(figsize=(10, 12))

california_boundary.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1,
)

listing_gdf.plot(
    ax=ax,
    markersize=4,
    alpha=0.4,
    legend=True,
)

ax.set_title("Listing Coordinates Over California Boundary")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

In [ ]:
sold_gdf = gpd.GeoDataFrame(
    sold, 
    geometry=gpd.points_from_xy(sold.Longitude, sold.Latitude))

fig, ax = plt.subplots(figsize=(10, 12))

california_boundary.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1,
)

sold_gdf.plot(
    ax=ax,
    markersize=4,
    alpha=0.4,
    legend=True,
)

ax.set_title("Sold Coordinates Over California Boundary")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

In [ ]:
listing = listing[listing["StateOrProvince"] == "CA"]
sold = sold[sold["StateOrProvince"] == "CA"]

In [ ]:
valid_zip_codes = set(
    pd.read_csv(
        "data/zip/california_valid_zip_codes.csv",
        dtype={"ZIP_CODE": "string"},
    )["ZIP_CODE"]
)

def filter_zip_codes(df):
    postal_code = df["PostalCode"].astype("string").str.strip()

    zip5 = postal_code.str.extract(
        r"^(\d{5})(?:-\d{4})?$",
        expand=False,
    )

    valid = zip5.isin(valid_zip_codes)

    df = df.loc[valid].copy()
    df["PostalCode"] = zip5.loc[valid]
    return df

In [ ]:
listing = filter_zip_codes(listing)
sold = filter_zip_codes(sold)

In [ ]:
# Remove NaN cities
def remove_nan_cities(df):
    df = df[df["City"].notna()]
    return df

In [ ]:
listing = remove_nan_cities(listing)
sold = remove_nan_cities(sold)

In [ ]:
def flag_coordinates(df):
    df = df.dropna(subset=["Latitude", "Longitude"])
    
    df_points = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
        crs="EPSG:4326",
    )

    df = gpd.sjoin(
            df_points,
            california_boundary[["geometry"]],
            how="inner",
            predicate="within",
        )

    df["coordinates_in_california"] = (
        df["Latitude"].notna()
        & df["Longitude"].notna()
        & df["index_right"].notna()
    )

    return pd.DataFrame(df.drop(columns=["geometry", "index_right"]))

In [ ]:
listing = flag_coordinates(listing)
sold = flag_coordinates(sold)

In [ ]:
def filter_nono_values(df, columns):
    for column in columns:
        df = df[df[column] >= 0]
    return df

In [ ]:
listing_columns = ["DaysOnMarket", "BedroomsTotal", 
               "BathroomsTotalInteger"]

sold_columns = listing_columns + ["ClosePrice"]

listing = filter_nono_values(listing, listing_columns)
sold = filter_nono_values(sold, sold_columns)

In [ ]:
listing

### Setting living area lowerbound

In [ ]:
listing = listing[listing["LivingArea"] > 80]
sold = sold[sold["LivingArea"] > 80]

### Dropping redundant columns
- PropertyType: Assumed to be `Residential`
- MlsStatus: Constant `Closed` value
- ListingKey: Redundant with `ListingKeyNumeric`
- `BuyerAgencyCompensationType`
- `OriginatingSystemName`
- `OriginatingSystemSubName`
- `AttachedGarageYN`
- `FireplaceYN`

In [ ]:
sold.drop(columns=["PropertyType", "MlsStatus", "ListingKey", 
                   "BuyerAgencyCompensationType", "OriginatingSystemName", 
                   "OriginatingSystemSubName", "AttachedGarageYN",
                   "FireplaceYN"], inplace=True)

In [ ]:
sold.columns

In [ ]:
print(listing.shape[0])
print(sold.shape[0])

### row count check:
- listing: `616072 -> 533594`
- sold: `447964 -> 442952`

### Basic analysis

In [ ]:
analysis_columns = [
    "OriginalListPrice", "ListPrice", "LivingArea", "DaysOnMarket",
    "LotSizeAcres", "YearBuilt", "BedroomsTotal",
    "BathroomsTotalInteger"
]
sold_analysis_columns = analysis_columns + ["ClosePrice"]

def basic_stats(df, columns):
    stats = df[columns].describe().T
    iqr = stats["75%"] - stats["25%"]
    lower = stats["25%"] - 1.5 * iqr
    upper = stats["75%"] + 1.5 * iqr
    stats["NaN"] = df[columns].isna().sum()
    stats["IQR outliers"] = (df[columns].lt(lower) | df[columns].gt(upper)).sum()
    return stats

### NaN counts

In [ ]:
listing.isna().sum().sort_values(ascending=False).to_frame("listing NaN")

In [ ]:
sold.isna().sum().sort_values(ascending=False).to_frame("sold NaN")

### Statistics and outliers

In [ ]:
basic_stats(listing, analysis_columns).rename_axis("listing")

In [ ]:
basic_stats(sold, sold_analysis_columns).rename_axis("sold")

### Distributions
Values are clipped at the 99th percentile for readability.

In [ ]:
listing[analysis_columns].clip(
    upper=listing[analysis_columns], axis=1
).hist(bins=30, figsize=(12, 8))
plt.suptitle("Listing distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
listing[analysis_columns].clip(
    upper=listing[analysis_columns].quantile(0.99), axis=1
).hist(bins=30, figsize=(12, 8))
plt.suptitle("Listing distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
sold[sold_analysis_columns].clip(
    upper=sold[sold_analysis_columns], axis=1
).hist(bins=30, figsize=(12, 8))
plt.suptitle("Sold distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
sold[sold_analysis_columns].clip(
    upper=sold[sold_analysis_columns].quantile(0.99), axis=1
).hist(bins=30, figsize=(12, 8))
plt.suptitle("Sold distributions", y=1.02)
plt.tight_layout()
plt.show()